# Etapa 3 — Probabilidade, Bayes e independência

Eventos atômicos, marginais, condicionais, regra da cadeia, independência e interpretação diagnóstica, sobre um recorte de treinamento que não consulta os meses de teste para calibrar a classe.

**Fonte:** BancoVDE 2026.xlsx / MJSP. **Semente:** 42.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_root():
    """Localiza a raiz do projeto (pasta com scripts/utils.py e data/) a partir do diretório atual."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "scripts" / "utils.py").exists() and (base / "data").exists():
            return base
    for pattern in ("*/scripts/utils.py", "*/*/scripts/utils.py"):   # ex.: Colab em /content
        for hit in sorted(here.glob(pattern)):
            if (hit.parent.parent / "data").exists():
                return hit.parent.parent
    raise FileNotFoundError(
        "Raiz do projeto não encontrada: abra o notebook com o diretório atual "
        "dentro da pasta FIA_P3_SegurancaPublica_Equipe5."
    )

ROOT = _find_root()

sys.path.insert(0, str(ROOT / "scripts"))
sys.modules.pop("utils", None)

from utils import (
    panel,
    training_split,
    prepare_splits,
    temporal_split,
    write_metrics
)

np.random.seed(42)

(ROOT / "outputs").mkdir(exist_ok=True)

print("Configuração da Etapa 3 concluída!")

Configuração da Etapa 3 concluída!


In [2]:
p, tr, te, _ = training_split()

A = tr.alto.astype(int)
B = tr.alta_previa.astype(int)
C = tr.semestre2.astype(int)

j = pd.crosstab(
    A,
    B,
    normalize=True
).reindex(
    index=[0, 1],
    columns=[0, 1],
    fill_value=0
)

print("Distribuição conjunta A x B:")
print(j)

assert abs(j.to_numpy().sum() - 1) < 1e-12

pa = float(A.mean())
pb = float(B.mean())

pab = float(
    ((A == 1) & (B == 1)).mean()
)

p_a_given_b = pab / pb

print("\nProbabilidades:")
print("P(A):", pa)
print("P(B):", pb)
print("P(A ∩ B):", pab)
print("P(A | B):", p_a_given_b)

Distribuição conjunta A x B:
alta_previa         0         1
alto                           
0            0.534266  0.241313
1            0.136100  0.088320

Probabilidades:
P(A): 0.22442084942084942
P(B): 0.3296332046332046
P(A ∩ B): 0.08832046332046332
P(A | B): 0.26793557833089315


## Axiomas, regra da cadeia e Bayes
Os quatro eventos atômicos somam exatamente 1, são disjuntos dois a dois, e $P(A\cup B)=P(A)+P(B)-P(A\cap B)$. A identidade de Bayes pode ser interpretada como reversão diagnóstica de uma condição, não uma influência causal.

In [3]:
pbc = float(((B == 1) & (C == 1)).mean())
pc = float(C.mean())

pabc = float(
    ((A == 1) & (B == 1) & (C == 1)).mean()
)

cond_a_bc = pabc / pbc
cond_b_c = pbc / pc

p_b_a = pab / pa
bayes = p_b_a * pa / pb

print("Regra da cadeia P(A ∩ B ∩ C):")
print(pabc, "=", cond_a_bc, "x", cond_b_c, "x", pc)
print("Resultado:", cond_a_bc * cond_b_c * pc)

print("\nTeorema de Bayes:")
print("P(A | B):", bayes)
print("Valor empírico:", pab / pb)

print("\nIndependência marginal:")
print("P(A ∩ B):", pab)
print("P(A) * P(B):", pa * pb)

print("\nIndependência condicional:")

for sem, g in tr.groupby("semestre2"):
    diff = float(
        ((g.alto == 1) & (g.alta_previa == 1)).mean()
        - g.alto.mean() * g.alta_previa.mean()
    )

    print(
        "Desvio de independência A e B dado C =",
        sem, ":", diff
    )

metrics = {
    "joint": j.to_numpy().tolist(),
    "joint_sum": float(j.to_numpy().sum()),
    "p_a": pa,
    "p_b": pb,
    "p_ab": pab,
    "p_a_given_b": bayes,
    "p_abc": pabc,
    "p_union": pa + pb - pab
}

write_metrics("etapa3.json", metrics)

Regra da cadeia P(A ∩ B ∩ C):
0.032335907335907334 = 0.23426573426573424 x 0.37336814621409925 x 0.3696911196911197
Resultado: 0.032335907335907334

Teorema de Bayes:
P(A | B): 0.26793557833089315
Valor empírico: 0.26793557833089315

Independência marginal:
P(A ∩ B): 0.08832046332046332
P(A) * P(B): 0.07397656378110046

Independência condicional:
Desvio de independência A e B dado C = 0 : 0.01689868178204494
Desvio de independência A e B dado C = 1 : 0.011428941502089465


PosixPath('FIA_P3_SegurancaPublica_Equipe5/outputs/etapa3.json')

## Limites da lógica clássica e da interpretação causal
A LPO é adequada para codificar regras `Se então` e relações entre entidades, mas a forma clássica não carrega a probabilidade `0,70` de uma proposição. Essa incerteza exige um modelo probabilístico adicional. Frequências de registros dependem de notificação, classificação e completude; correlações administrativas não são relações causais comprovadas.

## Independência condicional de dois atributos dada a classe (exigência do roteiro)
A tarefa pede especificamente examinar se `alta_previa` e `semestre2` são independentes quando se conhece a classe `alto`. A condição é $P(B,C\mid A)=P(B\mid A)P(C\mid A)$. Estimamos os dois lados separadamente em cada classe, sem interpretar pequenas diferenças como causalidade.

In [4]:
import json

cond = []

for cls, sub in tr.groupby("alto"):
    b = sub.alta_previa == 1
    c = sub.semestre2 == 1

    pbc = float((b & c).mean())
    prod = float(b.mean() * c.mean())

    cond.append({
        "classe_alto": int(cls),
        "p_bc_dada_classe": pbc,
        "p_b_c_dada_classe_produto": prod,
        "desvio": pbc - prod
    })

print(pd.DataFrame(cond).to_string(index=False))

m = json.loads(
    (ROOT / "outputs" / "etapa3.json").read_text()
)

m["independencia_B_C_dada_classe_A"] = cond

write_metrics("etapa3.json", m)

 classe_alto  p_bc_dada_classe  p_b_c_dada_classe_produto   desvio
           0          0.136279                   0.118105 0.018174
           1          0.144086                   0.132029 0.012057


PosixPath('FIA_P3_SegurancaPublica_Equipe5/outputs/etapa3.json')